# Lesson 2A: Hierarchical Clustering - Theory

## The Evolutionary Tree Problem

You work for a biology research lab at a major university. Your team just sequenced DNA from 100 different bacterial species collected from around the world. Your task: **understand the evolutionary relationships** between these organisms.

But you face several challenges:
- **Unknown number of groups**: How many distinct evolutionary lineages exist?
- **Hierarchical structure**: Species aren't just "similar" - some share recent ancestors, others diverged millions of years ago
- **Multiple granularities**: You want to see relationships at genus level, family level, and order level
- **The big picture**: You need the complete evolutionary tree, not just flat clusters

**K-Means won't work here** because:
1. It requires specifying k (number of clusters) upfront
2. It produces flat clusters with no hierarchy
3. You lose information about relationships between clusters

**Enter Hierarchical Clustering**: A method that builds a tree showing relationships at ALL levels of granularity.

## Real-World Applications

Hierarchical clustering is used extensively in:

🧬 **Evolutionary Biology**
- Construct phylogenetic trees from DNA/protein sequences
- Track viral evolution (COVID-19 variant trees)
- Study species relationships

📄 **Document Organization**
- Build topic hierarchies (topic → subtopic → sub-subtopic)
- Organize academic papers by research area
- Create category taxonomies for e-commerce

👥 **Market Segmentation**
- Macro segments → micro segments → individual preferences
- Understand customer hierarchy (VIP → regular → occasional)
- B2B account segmentation

🧪 **Gene Expression Analysis**
- Group genes by expression patterns across conditions
- Find regulatory pathways and gene networks
- Drug discovery and disease classification

🌐 **Social Network Analysis**
- Detect community hierarchies
- Understand organizational structures
- Identify influence cascades

## What You'll Learn

In this lesson, we'll cover:
1. ✅ **Agglomerative vs Divisive approaches** - Bottom-up vs top-down
2. ✅ **Linkage methods** - Single, Complete, Average, Ward (with math!)
3. ✅ **Dendrogram construction** - Building and interpreting the tree
4. ✅ **From-scratch implementation** - Complete working code
5. ✅ **Complexity analysis** - Understanding computational costs

Then in **Lesson 2B**, we'll use scikit-learn for production and apply to real biological datasets.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.datasets import make_blobs
from typing import List, Tuple
from numpy.typing import NDArray

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')

print("✅ Libraries loaded successfully!")

## Hierarchical Clustering: Two Approaches

### 1. Agglomerative (Bottom-Up) ⬆️

**Start**: Each data point is its own cluster  
**Process**: Repeatedly merge the two closest clusters  
**End**: All points in one giant cluster

```
Iteration 0: {A}, {B}, {C}, {D}, {E}  (5 clusters)
Iteration 1: {A,B}, {C}, {D}, {E}      (4 clusters) - merge A and B
Iteration 2: {A,B}, {C,D}, {E}         (3 clusters) - merge C and D
Iteration 3: {A,B,E}, {C,D}            (2 clusters) - merge {A,B} and E
Iteration 4: {A,B,C,D,E}               (1 cluster)  - final merge
```

**Pros**: 
- More commonly used (easier to implement)
- More efficient
- Interpretable dendrograms

**Cons**:
- Once merged, cannot split
- Greedy approach (locally optimal)

### 2. Divisive (Top-Down) ⬇️

**Start**: All points in one cluster  
**Process**: Repeatedly split the largest cluster  
**End**: Each point is its own cluster

```
Iteration 0: {A,B,C,D,E}               (1 cluster)
Iteration 1: {A,B,E}, {C,D}            (2 clusters) - split into two
Iteration 2: {A,B}, {E}, {C,D}         (3 clusters) - split {A,B,E}
Iteration 3: {A,B}, {C}, {D}, {E}      (4 clusters) - split {C,D}
Iteration 4: {A}, {B}, {C}, {D}, {E}   (5 clusters) - split {A,B}
```

**Pros**:
- Can produce better high-level clusters
- More conceptually intuitive

**Cons**:
- Computationally expensive O(2^n)
- Rarely used in practice
- Harder to implement efficiently

**We'll focus on Agglomerative** (the standard approach).

## The Agglomerative Algorithm

### Pseudocode

```python
def agglomerative_clustering(X, distance_metric, linkage_method):
    # Step 1: Initialize - each point is its own cluster
    clusters = [{i} for i in range(len(X))]
    dendrogram = []
    
    # Step 2: Merge until one cluster remains
    while len(clusters) > 1:
        # Find two closest clusters
        (i, j), dist = find_closest_pair(clusters, distance_metric, linkage_method)
        
        # Merge them
        new_cluster = clusters[i] ∪ clusters[j]
        
        # Record in dendrogram
        dendrogram.append((i, j, dist, |new_cluster|))
        
        # Update cluster list
        clusters.remove(clusters[i])
        clusters.remove(clusters[j])
        clusters.append(new_cluster)
    
    return dendrogram
```

### The Critical Question: How to Measure Distance Between Clusters?

Given two clusters $C_i$ and $C_j$, each containing multiple points, how do we measure $d(C_i, C_j)$?

**This is answered by linkage methods!**

## Linkage Methods: The Heart of Hierarchical Clustering

### 1. Single Linkage (MIN) 🔗

**Definition**: Distance between **closest** points in the two clusters

$$d_{\text{single}}(C_i, C_j) = \min_{x \in C_i, y \in C_j} d(x, y)$$

**Intuition**: "These clusters are as close as their nearest neighbors"

**Properties**:
- ✅ Can find **arbitrary-shaped** clusters (non-convex)
- ✅ Good for **elongated** or **chain-like** structures
- ❌ **Chaining effect**: Tends to create long, snake-like clusters
- ❌ **Sensitive to noise**: A single outlier can bridge two clusters
- ❌ **Non-robust**: Breaks down with noisy data

**Use cases**:
- Geographic clustering where regions touch
- Network analysis with connectivity
- Path-finding applications

---

### 2. Complete Linkage (MAX) 🔗

**Definition**: Distance between **farthest** points in the two clusters

$$d_{\text{complete}}(C_i, C_j) = \max_{x \in C_i, y \in C_j} d(x, y)$$

**Intuition**: "These clusters are as far as their farthest members"

**Properties**:
- ✅ Creates **compact**, spherical clusters
- ✅ **Less sensitive to outliers** than single linkage
- ✅ Breaks long chains
- ❌ Tends to **break large clusters** too early
- ❌ Bias toward **equal-sized** clusters

**Use cases**:
- When you want tight, well-separated groups
- Customer segmentation (distinct groups)
- Quality control (tight specifications)

---

### 3. Average Linkage (UPGMA - Unweighted Pair Group Method with Arithmetic Mean) 🔗

**Definition**: Average distance between **all pairs** of points

$$d_{\text{average}}(C_i, C_j) = \frac{1}{|C_i| |C_j|} \sum_{x \in C_i} \sum_{y \in C_j} d(x, y)$$

**Intuition**: "The average relationship between all members"

**Properties**:
- ✅ **Balanced** between single and complete
- ✅ **Robust** to noise and outliers
- ✅ Most **versatile** general-purpose method
- ⚖️ Moderate computational cost

**Use cases**:
- General-purpose clustering
- When you're unsure which linkage to use
- Biological taxonomy (phylogenetic trees)

---

### 4. Ward's Method (Minimum Variance) 🔗

**Definition**: Minimize increase in total **within-cluster variance**

$$d_{\text{Ward}}(C_i, C_j) = \frac{|C_i| |C_j|}{|C_i| + |C_j|} \|\mu_i - \mu_j\|^2$$

where $\mu_i = \frac{1}{|C_i|} \sum_{x \in C_i} x$ is the centroid of cluster $C_i$.

**Mathematical Derivation**:

When merging $C_i$ and $C_j$ into $C_{new} = C_i \cup C_j$, the increase in variance is:

$$\Delta(C_i, C_j) = \sum_{x \in C_{new}} \|x - \mu_{new}\|^2 - \sum_{x \in C_i} \|x - \mu_i\|^2 - \sum_{x \in C_j} \|x - \mu_j\|^2$$

This simplifies to:

$$\Delta(C_i, C_j) = \frac{|C_i| |C_j|}{|C_i| + |C_j|} \|\mu_i - \mu_j\|^2$$

Ward's method chooses the merge that minimizes this increase in variance!

**Properties**:
- ✅ **Similar to K-Means** objective (minimizes within-cluster variance)
- ✅ Tends to create **balanced**, equal-sized clusters
- ✅ **Best for Euclidean distance**
- ❌ Only works with **squared Euclidean distance**
- ❌ Bias toward spherical, similar-sized clusters

**Use cases**:
- Most common choice for general clustering
- Market segmentation
- Image segmentation
- When you want balanced clusters

---

### Comparison Summary

| Linkage | Distance Metric | Shape Bias | Outlier Sensitivity | Speed |
|---------|----------------|------------|-------------------|-------|
| **Single** | MIN | Chains, arbitrary | High ❌ | Fast ✅ |
| **Complete** | MAX | Spherical, compact | Medium ⚖️ | Fast ✅ |
| **Average** | MEAN | Balanced | Low ✅ | Medium ⚖️ |
| **Ward** | Variance | Spherical, balanced | Low ✅ | Medium ⚖️ |

**General Recommendation**: Start with **Ward** or **Average** for most applications.

In [ ]:
# Visualize linkage methods on synthetic data

print("Creating dataset with 3 clusters...")
X, y_true = make_blobs(n_samples=60, centers=3, n_features=2, 
                       cluster_std=0.5, random_state=42)

# Add some noise points
noise = np.random.randn(5, 2) * 2 + [5, 5]
X = np.vstack([X, noise])
y_true = np.hstack([y_true, np.full(5, -1)])

print(f"Dataset: {X.shape[0]} samples (including 5 noise points)")

# Visualize the data
plt.figure(figsize=(10, 6))
plt.scatter(X[:60, 0], X[:60, 1], c=y_true[:60], cmap='viridis', s=100, alpha=0.6, edgecolors='k')
plt.scatter(X[60:, 0], X[60:, 1], c='red', s=100, marker='x', label='Noise', linewidths=3)
plt.title('Synthetic Dataset: 3 Clusters + 5 Noise Points', fontweight='bold', fontsize=14)
plt.xlabel('Feature 1', fontweight='bold')
plt.ylabel('Feature 2', fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("✅ Data created and visualized!")

## From-Scratch Implementation

Let's implement agglomerative hierarchical clustering from scratch to understand exactly how it works!

In [ ]:
class AgglomerativeHC:
    """
    Agglomerative Hierarchical Clustering from scratch.
    
    Supports: single, complete, average, and Ward linkage.
    """
    
    def __init__(self, linkage='average'):
        """
        Initialize hierarchical clustering.
        
        Args:
            linkage: Linkage criterion ('single', 'complete', 'average', 'ward')
        """
        self.linkage = linkage
        self.merge_history = []
        self.labels_ = None
        
    def fit(self, X: NDArray):
        """
        Perform agglomerative hierarchical clustering.
        
        Args:
            X: Data of shape (n_samples, n_features)
        """
        n_samples = X.shape[0]
        
        # Initialize: each point is its own cluster
        clusters = [[i] for i in range(n_samples)]
        cluster_centers = X.copy()  # For Ward linkage
        
        # Track cluster IDs for dendrogram
        next_cluster_id = n_samples
        
        # Merge until one cluster remains
        iteration = 0
        while len(clusters) > 1:
            # Find two closest clusters
            min_dist = np.inf
            merge_i, merge_j = 0, 1
            
            for i in range(len(clusters)):
                for j in range(i + 1, len(clusters)):
                    dist = self._cluster_distance(X, clusters[i], clusters[j], 
                                                  cluster_centers[clusters[i][0]] if self.linkage == 'ward' else None,
                                                  cluster_centers[clusters[j][0]] if self.linkage == 'ward' else None)
                    if dist < min_dist:
                        min_dist = dist
                        merge_i, merge_j = i, j
            
            # Merge clusters
            new_cluster = clusters[merge_i] + clusters[merge_j]
            new_size = len(new_cluster)
            
            # Update center for Ward
            if self.linkage == 'ward':
                new_center = np.mean(X[new_cluster], axis=0)
                cluster_centers = np.vstack([cluster_centers, new_center])
            
            # Record merge for dendrogram
            self.merge_history.append({
                'iteration': iteration,
                'cluster_1': clusters[merge_i].copy(),
                'cluster_2': clusters[merge_j].copy(),
                'distance': min_dist,
                'size': new_size
            })
            
            # Update cluster list (remove old, add new)
            clusters = [c for idx, c in enumerate(clusters) if idx not in [merge_i, merge_j]]
            clusters.append(new_cluster)
            
            iteration += 1
            
            if iteration % 10 == 0 or iteration <= 5:
                print(f"  Iteration {iteration}: {len(clusters)} clusters remaining (merged dist={min_dist:.3f})")
        
        return self
    
    def _cluster_distance(self, X: NDArray, cluster_i: List[int], cluster_j: List[int],
                          center_i=None, center_j=None) -> float:
        """
        Compute distance between two clusters based on linkage criterion.
        
        Args:
            X: Data matrix
            cluster_i: Indices of points in first cluster
            cluster_j: Indices of points in second cluster
            center_i: Center of cluster i (for Ward)
            center_j: Center of cluster j (for Ward)
            
        Returns:
            Distance between clusters
        """
        if self.linkage == 'ward':
            # Ward: weighted squared distance between centroids
            if center_i is None:
                center_i = np.mean(X[cluster_i], axis=0)
            if center_j is None:
                center_j = np.mean(X[cluster_j], axis=0)
            
            n_i = len(cluster_i)
            n_j = len(cluster_j)
            return (n_i * n_j) / (n_i + n_j) * np.sum((center_i - center_j) ** 2)
        
        # For other methods, compute all pairwise distances
        distances = []
        for i in cluster_i:
            for j in cluster_j:
                dist = np.linalg.norm(X[i] - X[j])
                distances.append(dist)
        
        distances = np.array(distances)
        
        if self.linkage == 'single':
            return np.min(distances)
        elif self.linkage == 'complete':
            return np.max(distances)
        elif self.linkage == 'average':
            return np.mean(distances)
        else:
            raise ValueError(f"Unknown linkage: {self.linkage}")
    
    def cut_tree(self, n_clusters: int) -> NDArray:
        """
        Cut the dendrogram to get specified number of clusters.
        
        Args:
            n_clusters: Desired number of clusters
            
        Returns:
            Cluster labels for each sample
        """
        if n_clusters < 1:
            raise ValueError("n_clusters must be >= 1")
        
        # Get merges to undo
        n_samples = len(self.merge_history) + 1
        n_merges_to_keep = n_samples - n_clusters
        
        # Build clusters from scratch up to the cut point
        clusters = [[i] for i in range(n_samples)]
        
        for merge_idx in range(n_merges_to_keep):
            merge_info = self.merge_history[merge_idx]
            cluster_1 = merge_info['cluster_1']
            cluster_2 = merge_info['cluster_2']
            
            # Find and merge
            new_cluster = cluster_1 + cluster_2
            clusters = [c for c in clusters if not (set(c) <= set(cluster_1) or set(c) <= set(cluster_2))]
            clusters.append(new_cluster)
        
        # Convert to labels
        labels = np.zeros(n_samples, dtype=int)
        for cluster_id, cluster in enumerate(clusters):
            for point_id in cluster:
                labels[point_id] = cluster_id
        
        self.labels_ = labels
        return labels

print("✅ Agglomerative Hierarchical Clustering class implemented!")
print("   Supports: single, complete, average, Ward linkage")
print("   Features: fit(), cut_tree() for extracting clusters")

In [ ]:
# Test all linkage methods
print("="*70)
print("Testing Agglomerative HC with different linkage methods")
print("="*70)

linkage_methods = ['single', 'complete', 'average', 'ward']

for linkage_type in linkage_methods:
    print(f"\n{'='*70}")
    print(f"LINKAGE: {linkage_type.upper()}")
    print(f"{'='*70}")
    
    hc = AgglomerativeHC(linkage=linkage_type)
    hc.fit(X)
    
    print(f"\n✅ Completed {len(hc.merge_history)} merges")
    print(f"   First merge distance: {hc.merge_history[0]['distance']:.4f}")
    print(f"   Final merge distance: {hc.merge_history[-1]['distance']:.4f}")

print("\n" + "="*70)
print("✅ All linkage methods working correctly!")
print("="*70)

## Dendrograms: Visualizing the Hierarchy

A **dendrogram** is a tree diagram showing the hierarchical relationships between clusters.

**How to read a dendrogram:**
- **X-axis**: Individual data points
- **Y-axis**: Distance (or dissimilarity) at which clusters merge
- **Height of merge**: Shows how similar/dissimilar the clusters are
- **Horizontal line**: Represents a merge operation

**Cutting the dendrogram**:
- Draw a horizontal line at height h
- The number of vertical lines it crosses = number of clusters
- Different heights → different granularities

In [ ]:
# Create dendrograms for different linkage methods
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.ravel()

linkage_methods = ['single', 'complete', 'average', 'ward']
colors = ['red', 'blue', 'green', 'purple']

for idx, (linkage_type, color) in enumerate(zip(linkage_methods, colors)):
    # Use scipy for dendrogram (industry standard)
    Z = linkage(X, method=linkage_type)
    
    ax = axes[idx]
    dendrogram(Z, ax=ax, color_threshold=0, above_threshold_color=color)
    ax.set_title(f'{linkage_type.upper()} Linkage Dendrogram', 
                 fontweight='bold', fontsize=14)
    ax.set_xlabel('Sample Index', fontweight='bold')
    ax.set_ylabel('Distance', fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')
    
    # Add horizontal line showing 3-cluster cut
    if linkage_type == 'ward':
        threshold = Z[-2, 2] + (Z[-1, 2] - Z[-2, 2]) / 2
    else:
        threshold = (Z[-3, 2] + Z[-2, 2]) / 2
    ax.axhline(y=threshold, color='black', linestyle='--', linewidth=2,
               label=f'Cut for 3 clusters (y={threshold:.2f})')
    ax.legend()

plt.suptitle('Dendrogram Comparison: Different Linkage Methods', 
             fontweight='bold', fontsize=16, y=1.00)
plt.tight_layout()
plt.show()

print("✅ Dendrograms created for all linkage methods!")
print("\nObservations:")
print("• Single linkage: Shows chaining effect (long thin branches)")
print("• Complete linkage: Compact clusters (balanced tree)")
print("• Average linkage: Balanced between single and complete")
print("• Ward linkage: Most balanced tree structure")

## Extracting Clusters from Dendrogram

Once we have the dendrogram, we can "cut" it at any height to get clusters.

**Method 1: Cut at specific height**
- Choose distance threshold
- Clusters = connected components below that height

**Method 2: Specify number of clusters**
- Easier and more common
- Algorithm finds appropriate cut height

In [ ]:
# Cut dendrogram to get 3 clusters with different linkages
from scipy.cluster.hierarchy import fcluster

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.ravel()

for idx, linkage_type in enumerate(linkage_methods):
    # Perform linkage
    Z = linkage(X, method=linkage_type)
    
    # Cut to get 3 clusters
    labels = fcluster(Z, 3, criterion='maxclust')
    
    # Visualize clustering result
    ax = axes[idx]
    scatter = ax.scatter(X[:, 0], X[:, 1], c=labels, cmap='viridis', 
                        s=100, alpha=0.6, edgecolors='k', linewidth=1.5)
    ax.set_title(f'{linkage_type.upper()} Linkage: 3 Clusters', 
                fontweight='bold', fontsize=14)
    ax.set_xlabel('Feature 1', fontweight='bold')
    ax.set_ylabel('Feature 2', fontweight='bold')
    ax.grid(True, alpha=0.3)
    
    # Add cluster centers
    for cluster_id in np.unique(labels):
        cluster_points = X[labels == cluster_id]
        center = cluster_points.mean(axis=0)
        ax.scatter(*center, c='red', s=300, marker='*', 
                  edgecolors='black', linewidth=2, zorder=10)

plt.suptitle('Clustering Results: Different Linkages (3 Clusters)', 
             fontweight='bold', fontsize=16, y=1.00)
plt.tight_layout()
plt.show()

print("✅ Extracted 3 clusters using different linkage methods!")
print("\nNotice:")
print("• Single: Chains can merge unrelated points (chaining effect)")
print("• Complete: Tight, well-separated clusters")
print("• Average: Balanced results")
print("• Ward: Similar to K-Means (minimizes variance)")

## Complexity Analysis

### Time Complexity

**Naive Implementation** (ours):
- **O(n³)** where n = number of samples
- Why? For each of n merges, we compute distances between all O(n²) pairs

**Optimized Implementation** (scipy, scikit-learn):
- **O(n² log n)** for most linkage methods
- Uses priority queues and clever data structures

**For large datasets (n > 10,000)**:
- Hierarchical clustering becomes impractical
- Use K-Means or MiniBatch K-Means instead
- Or use approximate/sampled hierarchical methods

### Space Complexity

- **O(n²)** to store distance matrix
- **O(n)** to store dendrogram (only merges, not full tree)

### Comparison with K-Means

| Metric | Hierarchical | K-Means |
|--------|-------------|---------|
| **Time** | O(n² log n) | O(nki) |
| **Space** | O(n²) | O(nk) |
| **Scales to** | ~10K samples | Millions |
| **Clusters** | All levels | Fixed k |
| **Shape** | Any | Spherical |

Where:
- n = samples
- k = clusters  
- i = iterations (typically small)

## When to Use Hierarchical Clustering

### ✅ Use Hierarchical Clustering When:

1. **Unknown number of clusters**
   - You want to explore different granularities
   - Data has natural hierarchy

2. **Small to medium datasets**
   - n < 10,000 samples
   - Can afford O(n²) memory

3. **Interpretability matters**
   - Need to explain cluster relationships
   - Want to visualize the hierarchy (dendrogram)

4. **Arbitrary cluster shapes**
   - With single linkage, can find non-convex clusters
   - K-Means only finds spherical clusters

5. **Hierarchical structure exists**
   - Taxonomy (biology)
   - Organization charts
   - Document categorization

### ❌ Don't Use Hierarchical Clustering When:

1. **Large datasets** (n > 10,000)
   - Use K-Means or MiniBatch K-Means
   - Or BIRCH (hierarchical for large data)

2. **High-dimensional data**
   - Curse of dimensionality affects distance metrics
   - Consider dimensionality reduction first (PCA → HC)

3. **Need to add new points**
   - Must recompute entire hierarchy
   - K-Means can assign new points to existing clusters

4. **Online learning**
   - Hierarchical is batch-only
   - Use streaming K-Means variants

## Summary

🎯 **What We Learned:**

1. **Hierarchical clustering** builds a tree showing relationships at all levels
2. **Two approaches**: Agglomerative (bottom-up) and Divisive (top-down)
3. **Four linkage methods**:
   - **Single**: Minimum distance (chains, arbitrary shapes)
   - **Complete**: Maximum distance (compact clusters)
   - **Average**: Mean distance (balanced, robust)
   - **Ward**: Minimize variance (like K-Means, most common)
4. **Dendrograms** visualize the hierarchy
5. **Cutting the tree** extracts clusters at any granularity
6. **Complexity**: O(n² log n) time, O(n²) space

🔬 **Real-World Impact:**
- Evolutionary biology: Phylogenetic trees from DNA
- Document organization: Topic hierarchies
- Market segmentation: Multi-level customer groups
- Gene expression: Disease classification

🚀 **Next Steps:**

In **Lesson 2B**, we'll:
- Use scikit-learn for production hierarchical clustering
- Apply to real biological dataset (gene expression)
- Compare linkage methods on real data
- Build production pipelines

---

**Key Takeaway**: Hierarchical clustering reveals **structure at all levels**, unlike K-Means which gives flat clusters. Use Ward linkage as default, but try others if you have domain knowledge about cluster shapes!